# Smart Intersection: training both networks on Google Colab

**Before you start:** in the menu **Runtime → Change runtime type** choose **T4 GPU** and save.

At the end, download `counter_cnn.pt` and `traffic_dqn.pt` into the project's `models/` folder on your computer, and run the app there.

## 1. Upload the project
Choose the project zip file.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name"
import glob, os
# the project folder: the one with src/make_camera_data.py (works for the project zip or the whole repository)
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(sorted(glob.glob('**/src/make_camera_data.py', recursive=True), key=len)[0]))))
print('Project folder:', os.getcwd())
!ls

## 2. Check the GPU

In [ ]:
!nvidia-smi -L
import torch, os
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available(), '| CPU cores:', os.cpu_count())

## 3. Data collection: 32,000 labeled camera images
The simulation "photographs" the intersection and labels every image automatically. This runs on the CPU, so it takes a few minutes.

In [ ]:
!python -m src.make_camera_data
from IPython.display import Image, display
display(Image('reports/camera_examples.png'))
display(Image('reports/camera_label_hist.png'))

## 4. Train network 1: counting cars
At the start of the run you should see the line `Device: cuda`.

In [ ]:
!python -m src.train_counter

In [ ]:
display(Image('reports/counter_curves.png'))

## 5. Evaluate network 1 on intersections not seen in training

In [ ]:
!python -m src.evaluate_counter
for f in ['counter_scatter', 'counter_examples']:
    display(Image(f'reports/{f}.png'))

## 6. Train network 2: the agent that manages the traffic light
The network is small, so training is fast even on a CPU.

In [ ]:
!python -m src.train_agent
display(Image('reports/agent_curves.png'))

## 7. Evaluate the whole system
Camera, network 1, network 2 and traffic light, vs. a regular traffic light, in day, dusk, night and rain. Takes a few minutes, because camera images are drawn at every decision.

In [ ]:
!python -m src.evaluate
for f in ['system_results', 'queue_timeline']:
    display(Image(f'reports/{f}.png'))

## 8. (Optional) Experiment: training the agent with counting noise
The agent trains with noise the size of network 1's error. Then compare `system_report.txt` before and after.

**Note:** this command replaces `traffic_dqn.pt`. If you want to keep both versions, download the previous one first.

In [ ]:
# !python -m src.train_agent --noise auto
# !python -m src.evaluate

## 9. Download the networks and reports
* Put `counter_cnn.pt` and `traffic_dqn.pt` into the `models/` folder of the project on your computer
* Extract the contents of `reports.zip` into the `reports/` folder

If the browser blocks multiple downloads, download from the files pane (the folder icon on the left).

In [ ]:
!zip -q -r reports.zip reports
files.download('models/counter_cnn.pt')
files.download('models/traffic_dqn.pt')
files.download('reports.zip')

## 10. (Optional) Export for the web page (GitHub Pages)
Packs both networks into `docs/model/` (and draws the camera background into `docs/scene.png`), so the page in `docs/` can run them in the browser. Extract `docs_model.zip` into the project folder on your computer, then publish (see the README).

In [ ]:
!python -m src.export_web
!zip -q -r docs_model.zip docs/model docs/scene.png docs/scene.json
files.download('docs_model.zip')